# Netflix ETL

Limpieza, enriquecimiento y carga relacional del catálogo de Netflix.

## Configuración

Carga variables desde `.env`, prepara la conexión a MySQL y habilita una caché local para las consultas HTTP.

In [ ]:
import hashlib
import pickle
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
import requests
import requests_cache
from dotenv import load_dotenv
from sqlalchemy import create_engine

BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "netflix_titles.csv"
TMDB_CACHE_PATH = BASE_DIR / ".cache" / "tmdb_http_cache"
ENRICHMENT_CACHE_PATH = BASE_DIR / ".cache" / "tmdb_enrichment.pkl"
TMDB_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

tmdb_session = requests_cache.CachedSession(
    str(TMDB_CACHE_PATH),
    backend="sqlite",
    expire_after=None,
    allowable_methods=("GET",),
    stale_if_error=True
)

load_dotenv(BASE_DIR / ".env", override=True)
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = os.getenv("MYSQL_PORT", "3306")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "netflix_etl")
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "")
DATABASE_URL = (
    f"mysql+pymysql://{quote_plus(MYSQL_USER)}:"
    f"{quote_plus(MYSQL_PASSWORD)}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

print(f"Dataset: {CSV_PATH.name} | API TMDB configurada: {bool(TMDB_API_KEY)}")

## Extract

Lee el dataset original sin modificarlo.

In [ ]:
# Cargar el dataset directamente desde CSV
df = pd.read_csv(CSV_PATH)

print(f"Dataset extraído: {df.shape[0]} filas, {df.shape[1]} columnas")
display(df.head())

## Transform

Normaliza texto, fechas y países. Si existe una clave de TMDB, completa director, reparto y país solo cuando faltan datos.

In [ ]:
import re
import unicodedata
from difflib import get_close_matches

COUNTRY_REFERENCE_URL = "https://countriesnow.space/api/v0.1/countries/positions"
country_reference_response = tmdb_session.get(COUNTRY_REFERENCE_URL, timeout=10)
country_reference_response.raise_for_status()
COUNTRY_REFERENCE_NAMES = sorted({
    item["name"].strip()
    for item in country_reference_response.json().get("data", [])
    if item.get("name")
})


def _country_key(value):
    normalized = unicodedata.normalize("NFKD", str(value))
    normalized = "".join(
        character for character in normalized
        if not unicodedata.combining(character)
    )
    return re.sub(r"[^a-z0-9]", "", normalized.casefold())


COUNTRY_CANONICAL_BY_KEY = {
    _country_key(country): country
    for country in COUNTRY_REFERENCE_NAMES
}
COUNTRY_ALIASES = {
    "usa": "United States",
    "us": "United States",
    "unitedstatesofamerica": "United States",
    "uk": "United Kingdom",
    "u k": "United Kingdom"
}


def normalize_country_name(country):
    """Convierte nombres y errores cercanos al nombre canónico de la API."""
    if not isinstance(country, str):
        return country
    key = _country_key(country)
    alias = COUNTRY_ALIASES.get(key)
    if alias:
        return alias
    canonical = COUNTRY_CANONICAL_BY_KEY.get(key)
    if canonical:
        return canonical
    close_keys = get_close_matches(key, COUNTRY_CANONICAL_BY_KEY, n=1, cutoff=0.86)
    return COUNTRY_CANONICAL_BY_KEY[close_keys[0]] if close_keys else country

print(f"Referencia de países cargada: {len(COUNTRY_REFERENCE_NAMES)} nombres canónicos")

In [ ]:
# ============================================================
# TRANSFORM: funciones reutilizables y pipeline
# ============================================================


def _normalize_country_values(countries):
    if not isinstance(countries, list):
        return countries
    normalized = [normalize_country_name(country) for country in countries]
    return list(dict.fromkeys(normalized))


def _normalize_country_column(dataframe):
    dataframe["country"] = dataframe["country"].apply(_normalize_country_values)
    return dataframe


def basic_cleanup(dataframe):
    """Aplica la limpieza original y valida el resultado."""
    cleaned = dataframe.copy()

    duration_in_rating = cleaned["rating"].str.match(r"^\d+\s+min$", na=False)
    cleaned.loc[duration_in_rating, "duration"] = cleaned.loc[
        duration_in_rating, "rating"
    ]
    cleaned.loc[duration_in_rating, "rating"] = pd.NA
    cleaned["date_added"] = pd.to_datetime(cleaned["date_added"], errors="coerce")

    text_columns = [
        "show_id", "type", "title", "director", "cast", "country",
        "rating", "duration", "listed_in", "description"
    ]
    for column in text_columns:
        cleaned[column] = cleaned[column].str.strip()

    cleaned[text_columns] = cleaned[text_columns].replace(r"^\s*$", pd.NA, regex=True)

    for column in ["director", "cast", "country", "listed_in"]:
        cleaned[column] = cleaned[column].apply(
            lambda value: [item.strip() for item in value.split(",")]
            if pd.notna(value) else pd.NA
        )

    _normalize_country_column(cleaned)
    cleaned["date_added"] = cleaned["date_added"].dt.date
    print(f"Limpieza completada: {cleaned.shape[0]} filas, {cleaned.shape[1]} columnas")
    print("Nulos principales:", cleaned[["director", "cast", "country"]].isna().sum().to_dict())
    print("Países normalizados con la API de países")
    return cleaned


def _tmdb_lookup(row, session):
    """Obtiene director, actores y países para una fila desde TMDB."""
    search_type = "movie" if row["type"] == "Movie" else "tv"
    search_response = session.get(
        f"https://api.themoviedb.org/3/search/{search_type}",
        params={"api_key": TMDB_API_KEY, "query": row["title"]},
        timeout=10
    )
    search_response.raise_for_status()
    results = search_response.json().get("results", [])
    if not results:
        return {"director": None, "cast": None, "country": None}

    tmdb_id = results[0]["id"]
    details_response = session.get(
        f"https://api.themoviedb.org/3/{search_type}/{tmdb_id}",
        params={"api_key": TMDB_API_KEY, "append_to_response": "credits"},
        timeout=10
    )
    details_response.raise_for_status()
    data = details_response.json()

    directors = [
        person["name"] for person in data.get("credits", {}).get("crew", [])
        if person.get("job") == "Director"
    ]
    actors = [
        person["name"] for person in data.get("credits", {}).get("cast", [])[:10]
    ]
    countries = _normalize_country_values([
        country["name"]
        for country in data.get("production_countries", [])
    ])
    return {
        "director": directors or None,
        "cast": actors or None,
        "country": countries or None
    }


def _enrichment_fingerprint(dataframe):
    """Genera una huella estable para detectar cambios en la entrada del enriquecimiento."""
    fingerprint_columns = ["show_id", "type", "title", "director", "cast", "country"]
    comparable = dataframe[fingerprint_columns].astype("string")
    values = pd.util.hash_pandas_object(comparable, index=True).values.tobytes()
    return hashlib.sha256(values).hexdigest()


def _load_enrichment_cache():
    """Carga la caché de resultados completos y consultas individuales."""
    if not ENRICHMENT_CACHE_PATH.exists():
        return {"version": 2, "dataframes": {}, "lookups": {}}

    try:
        with ENRICHMENT_CACHE_PATH.open("rb") as cache_file:
            cache = pickle.load(cache_file)
        cache.setdefault("lookups", {})
        if cache.get("version") != 2:
            cache = {"version": 2, "dataframes": {}, "lookups": cache["lookups"]}
        else:
            cache.setdefault("dataframes", {})
        return cache
    except (EOFError, OSError, ValueError, AttributeError, KeyError, pickle.UnpicklingError):
        return {"version": 2, "dataframes": {}, "lookups": {}}


def _save_enrichment_cache(cache):
    """Guarda la caché de forma atómica para conservarla si la ejecución se interrumpe."""
    temporary_path = ENRICHMENT_CACHE_PATH.with_suffix(".tmp")
    with temporary_path.open("wb") as cache_file:
        pickle.dump(cache, cache_file, protocol=pickle.HIGHEST_PROTOCOL)
    temporary_path.replace(ENRICHMENT_CACHE_PATH)


def enrich_from_api(dataframe):
    """Completa campos faltantes usando caché persistente y TMDB solo cuando es necesario."""
    enriched = dataframe.copy()
    _normalize_country_column(enriched)
    if not TMDB_API_KEY:
        print("API TMDB no configurada; se conserva el dataset sin enriquecimiento.")
        return enriched

    cache = _load_enrichment_cache()
    fingerprint = _enrichment_fingerprint(enriched)
    cached_dataframe = cache["dataframes"].get(fingerprint)
    if isinstance(cached_dataframe, pd.DataFrame):
        cached_dataframe = cached_dataframe.copy()
        _normalize_country_column(cached_dataframe)
        print(f"Caché completa reutilizada: {ENRICHMENT_CACHE_PATH}")
        return cached_dataframe

    pending = enriched[
        enriched[["director", "cast", "country"]].isna().any(axis=1)
    ].copy()
    pending["cache_type"] = pending["type"].map(
        lambda value: "movie" if value == "Movie" else "tv"
    )
    unique_pending = pending.drop_duplicates(subset=["cache_type", "title"])
    cache_hits = 0
    api_requests = 0

    print(f"Registros candidatos: {len(pending)}")
    print(f"Títulos únicos a consultar: {len(unique_pending)}")

    for _, row in unique_pending.iterrows():
        cache_key = (row["cache_type"], row["title"])
        if cache_key in cache["lookups"]:
            result = cache["lookups"][cache_key]
            cache_hits += 1
        else:
            try:
                result = _tmdb_lookup(row, tmdb_session)
                cache["lookups"][cache_key] = result
                api_requests += 1
                if api_requests % 25 == 0:
                    _save_enrichment_cache(cache)
            except (requests.exceptions.RequestException, KeyError, ValueError) as error:
                print(f"API no disponible para {row['title']}: {error}")
                continue

        result["country"] = _normalize_country_values(result.get("country"))
        matching_indexes = pending.index[
            (pending["cache_type"] == row["cache_type"])
            & (pending["title"] == row["title"])
        ]
        for index in matching_indexes:
            for column in ["director", "cast", "country"]:
                current_value = enriched.at[index, column]
                missing = not isinstance(current_value, list) and pd.isna(current_value)
                if missing and result.get(column):
                    enriched.at[index, column] = result[column]

    _normalize_country_column(enriched)
    cache["dataframes"][fingerprint] = enriched.copy()
    _save_enrichment_cache(cache)
    print(f"Resultados reutilizados desde caché: {cache_hits}")
    print(f"Consultas nuevas: {api_requests}")
    print(f"Caché persistente: {ENRICHMENT_CACHE_PATH}")
    print("Enriquecimiento API completado.")
    return enriched


df_clean = df.pipe(basic_cleanup).pipe(enrich_from_api)
display(df_clean.head())

## Validación

Revisa duplicados y valores faltantes antes de cargar los datos.

In [ ]:
# Resumen de calidad después de la transformación
validation = pd.Series({
    "filas": len(df_clean),
    "show_id duplicados": df_clean["show_id"].duplicated().sum(),
    "filas duplicadas": df_clean.astype(str).duplicated().sum(),
    "fechas nulas": df_clean["date_added"].isna().sum(),
    "director nulo": df_clean["director"].isna().sum(),
    "reparto nulo": df_clean["cast"].isna().sum(),
    "país nulo": df_clean["country"].isna().sum(),
})
print(validation)
display(df_clean.head())


## Load

Carga dimensiones, shows y tablas puente en el esquema MySQL configurado.

In [ ]:
from sqlalchemy import MetaData, Table, create_engine, insert


def _dimension(dataframe, source_column, target_column):
    values = dataframe[source_column].dropna().explode().dropna().astype(str).str.strip()
    return pd.DataFrame({target_column: values[values.ne("")].unique()})


def _bridge(dataframe, source_column, id_column, id_map):
    links = dataframe[["show_id", source_column]].dropna(subset=[source_column]).explode(source_column)
    links[source_column] = links[source_column].astype(str).str.strip()
    links = links[links[source_column].ne("")].rename(columns={source_column: "name"})
    links[id_column] = links["name"].map(id_map)
    return links[["show_id", id_column]].dropna().drop_duplicates()


def _insert_ignore(dataframe, table_name, database_engine):
    table = Table(table_name, MetaData(), autoload_with=database_engine)
    records = (
        dataframe.astype(object)
        .where(pd.notna(dataframe), None)
        .to_dict(orient="records")
    )
    if records:
        with database_engine.begin() as connection:
            connection.execute(insert(table).prefix_with("IGNORE"), records)


def _write_dimension(dataframe, table_name, name_column, id_column, database_engine):
    _insert_ignore(dataframe, table_name, database_engine)
    loaded = pd.read_sql_table(table_name, database_engine, columns=[id_column, name_column])
    return dict(zip(loaded[name_column], loaded[id_column]))


def load_relational_data(dataframe, database_engine):
    """Carga dimensiones, shows y relaciones según el esquema objetivo."""
    if database_engine is None:
        raise RuntimeError(
            "MySQL no está configurado. Define las variables MYSQL_* en .env "
            "antes de ejecutar la carga."
        )

    working = dataframe.copy()

    ratings = working[["rating"]].dropna().drop_duplicates().rename(columns={"rating": "rating_code"})
    _insert_ignore(ratings, "ratings", database_engine)
    ratings_loaded = pd.read_sql_table("ratings", database_engine)
    rating_map = dict(zip(ratings_loaded["rating_code"], ratings_loaded["rating_id"]))

    dimension_specs = [
        ("director", "directors", "name", "director_id"),
        ("cast", "actors", "name", "actor_id"),
        ("country", "countries", "name", "country_id"),
        ("listed_in", "genres", "name", "genre_id")
    ]
    maps = {}
    for source_column, table_name, name_column, id_column in dimension_specs:
        dimension = _dimension(working, source_column, name_column)
        maps[source_column] = _write_dimension(
            dimension, table_name, name_column, id_column, database_engine
        )

    duration_parts = working["duration"].astype("string").str.extract(
        r"(?P<duration_value>\d+)\s*(?P<duration_unit>min|season|seasons)?",
        expand=True
    )
    shows = working[[
        "show_id", "type", "title", "date_added", "release_year",
        "rating", "description"
    ]].copy()
    shows["rating_id"] = shows["rating"].map(rating_map)
    shows["duration_value"] = pd.to_numeric(duration_parts["duration_value"], errors="coerce")
    shows["duration_unit"] = duration_parts["duration_unit"].replace({"seasons": "season"})
    shows = shows.drop(columns=["rating"]).drop_duplicates("show_id")
    _insert_ignore(shows, "shows", database_engine)

    bridges = [
        ("director", "director_id", "show_director"),
        ("cast", "actor_id", "show_actor"),
        ("country", "country_id", "show_country"),
        ("listed_in", "genre_id", "show_genre")
    ]
    for source_column, id_column, table_name in bridges:
        _insert_ignore(
            _bridge(working, source_column, id_column, maps[source_column]),
            table_name,
            database_engine
        )

    print(f"Carga relacional completada: {len(shows)} shows")
    return shows


try:
    shows_loaded = load_relational_data(df_clean, engine)
    print(f"Carga a MySQL exitosa: {len(shows_loaded)} shows cargados.")
except Exception as error:
    print(f"Carga a MySQL fallida: {type(error).__name__}: {error}")